# MLP neuron screening — instruct models

Reproduces **Appendix N (Table 21)** of the paper: per-neuron directional DLA
on the cultural-binding task, continuous specificity ranking
`spec = |Δdir| / (leak + eps)` over the top-300 `|Δdir|` pool (top-200 kept),
and mean-ablation of the top-K toward-R (POS) and anti-R (NEG) neurons for
K ∈ {10, 20, 40}, with a neutral-prompt split A/B control (split seed 7):
split A supplies the leak term used for ranking, split B is the independent
post-ablation neutral control (non-circular).

The neuron pipeline (bootstrap + Cell 1 + Cell 2) together with the DLA
setup cells (per-head DLA + directional cells for pre-norm architectures
and Gemma-2).
The exec-by-index bootstrap is replaced by inline verbatim cells; the
architecture branch is selected at runtime from `CFG["has_softcapping"]`,
exactly like the bootstrap's `DLA_DIR[bool(CFG["has_softcapping"])]`
dispatch. Shared setup (config, data loading, chat formatting) now comes
from `common/`.

Run top-to-bottom with `MODEL_KEY` set to one of
`{mistral, llama, gemma2, nemo}`. Outputs, under
`./results/<MODEL_KEY>_instruct/`:

- `results_<MODEL_KEY>_instruct_dla.pkl` — per-head DLA maps (saved by the
  DLA cell, as in the source);
- `results_<MODEL_KEY>_instruct_mlp_neuron_filtered.pkl` — neuron ranking
  (`kept`, `delta_dir`, `leak`, `neutralB`), then updated in place by the
  ablation cell (`sweep`, `baseline`).

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
HF_TOKEN = config.HF_TOKEN          # read from the HF_TOKEN env var
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
ACTIVE_MODEL = config.ACTIVE_MODEL

from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import format_for_chat, find_option_token_ids

# Imported up front by the mlp_v2 bootstrap ("was missing before"): the
# directional cells below use ttest_1samp without importing it themselves.
from scipy.stats import ttest_1samp, ttest_rel

In [ ]:
import os
import re
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager
from itertools import combinations

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy import stats
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

In [ ]:
print("=" * 80)
print(f"STAGE 1: S-SCORES — {CFG['label']}")
print("=" * 80)

# ── Load data ──
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# ── Load model ──
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
config.model = model; config.tokenizer = tokenizer

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
config.first_device = first_device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# ── Format texts ──
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# ── Compute S-scores ──
print(f"\n  Computing S-scores...")
results_sscore = {}
for cond in conditions:
    scores, c_chosen, p_c_list = [], [], []
    for i, text in enumerate(texts_fmt[cond]):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
        scores.append(S)
        chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
        c_chosen.append(1 if chosen == 'c' else 0)
        lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
        p_c_list.append(np.exp(lp_c - lp_all))
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            print(f"    {cond}: {i}/{n_total}")
            torch.cuda.empty_cache()
    results_sscore[cond] = {
        'S': np.array(scores), 'c_chosen': np.array(c_chosen),
        'p_c': np.array(p_c_list),
    }

# ── Print results ──
print(f"\n{'':20s}  {'Mean S':>8s}  {'Med S':>8s}  {'(c) rate':>8s}  {'Mean P(c)':>9s}")
for cond in conditions:
    r = results_sscore[cond]
    print(f"  {cond:20s}  {r['S'].mean():8.3f}  {np.median(r['S']):8.3f}  "
          f"{r['c_chosen'].mean():8.3f}  {r['p_c'].mean():9.3f}")
delta_S = results_sscore['B_cult']['S'].mean() - results_sscore['B_unrel']['S'].mean()
print(f"\n  Δ(S) = {delta_S:.4f}")

## Per-head DLA — architecture dispatch

The two cells below are the per-head DLA compute cells (10 and 20);
exactly one runs, selected by `CFG["has_softcapping"]` (pre-norm:
mistral / llama / nemo; Gemma-2: `(1 + w)` norm scaling, per-layer frozen
post-attention norm, sanity checks in pre-softcap logit space). This replaces
the bootstrap's `DLA_DIR = {False: (10, 39), True: (20, 34)}` exec-by-index.
Each saves `results_<MODEL_KEY>_instruct_dla.pkl`.

In [ ]:
# ── Architecture dispatch (bootstrap: DLA_DIR[bool(CFG["has_softcapping"])]) — pre-norm branch ──
if not bool(CFG["has_softcapping"]):
    # ================================================================
    # DLA — DIRECT LOGIT ATTRIBUTION (compute)
    # Attention-pattern-independent head ranking, complementary to the
    # QK-conditioned discovery (attention features + edge KO). Captures each
    # head's o_proj input at the last token, maps it through W_O and the
    # frozen final norm, and projects onto the option reading direction.
    # Requires: cells 1-7 executed (model + data + texts_fmt + option_tokens).
    # ================================================================
    import time

    print("=" * 80)
    print(f"DLA: DIRECT LOGIT ATTRIBUTION — {CFG['label']}")
    print("=" * 80)

    n_layers = model.config.num_hidden_layers
    n_heads  = model.config.num_attention_heads
    head_dim = getattr(model.config, "head_dim", None) or model.config.hidden_size // n_heads
    d_model  = model.config.hidden_size
    print(f"  {n_layers} layers x {n_heads} heads, head_dim={head_dim}, d_model={d_model}")
    print(f"  (GQA only affects K/V; o_proj input is per-QUERY-head, so the "
          f"per-head decomposition at o_proj is exact)")

    # ── Reading direction: u = e_c - 0.5*(e_a + e_b) in unembedding space.
    # find_option_token_ids returns multiple ids per option (e.g. 'a' and 'A');
    # we average the corresponding unembedding rows per option.
    W_U = model.lm_head.weight.detach().float()
    e_opt = {opt: W_U[ids].mean(dim=0) for opt, ids in option_tokens_raw.items()}
    u = (e_opt['c'] - 0.5 * (e_opt['a'] + e_opt['b'])).to(first_device)
    del W_U, e_opt

    # ── Frozen-norm handling: DLA_{l,h} = u . ( c_{l,h} / rms(resid_final) * g )
    # where g = final RMSNorm weight and rms is computed from the REAL final
    # residual of each prompt (captured by a hook on model.model.norm input).
    # u and g are fixed, so fold them once: u_eff = u * g.
    norm_w   = model.model.norm.weight.detach().float()
    norm_eps = model.model.norm.variance_epsilon
    u_eff = u * norm_w

    # ── Fold u_eff through W_O per layer: u_eff . W_O[:, h*hd:(h+1)*hd] @ z_{l,h}
    # equals (W_O^T @ u_eff)[h-block] . z_{l,h}, so per prompt the full
    # [n_layers, n_heads] DLA map is one einsum against the captured z's —
    # no d_model-sized per-head contribution is ever materialized.
    U_proj = torch.empty(n_layers, n_heads, head_dim, device=first_device)
    for l in range(n_layers):
        W_O = model.model.layers[l].self_attn.o_proj.weight.detach().float()
        U_proj[l] = (W_O.T @ u_eff).view(n_heads, head_dim)
        del W_O
    print(f"  U_proj precomputed: {tuple(U_proj.shape)}")

    # ── Hooks: o_proj INPUT (z, per-query-head) + final-norm INPUT (residual),
    # last token position only.
    _cap = {'z': [None] * n_layers, 'resid': None}

    def _make_oproj_hook(l):
        def hook(module, args):
            _cap['z'][l] = args[0][0, -1, :].detach().float().view(n_heads, head_dim)
        return hook

    def _norm_hook(module, args):
        _cap['resid'] = args[0][0, -1, :].detach().float()

    def dla_forward(text):
        """One forward pass; returns per-head DLA map + S-score + sanity pieces."""
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        z = torch.stack(_cap['z'])                       # [n_layers, n_heads, head_dim]
        resid = _cap['resid']                            # [d_model]
        rms = torch.sqrt((resid ** 2).mean() + norm_eps)
        dla = (torch.einsum('lhk,lhk->lh', z, U_proj) / rms).cpu().numpy()

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()

        # Sanity pieces: u . norm_frozen(resid) must equal the same contrast read
        # directly off the final logits (frozen rms comes from the real residual,
        # so this checks u construction + norm handling end to end).
        u_dot_resid = ((u_eff @ resid) / rms).item()
        direct = (logits[option_tokens['c']].mean()
                  - 0.5 * (logits[option_tokens['a']].mean()
                           + logits[option_tokens['b']].mean())).item()
        out = {'dla': dla, 'S': S, 'rms': rms.item(),
               'u_dot_resid': u_dot_resid, 'direct': direct}
        del enc, outputs, logits, lp, z, resid
        return out

    hooks = [model.model.layers[l].self_attn.o_proj.register_forward_pre_hook(_make_oproj_hook(l))
             for l in range(n_layers)]
    hooks.append(model.model.norm.register_forward_pre_hook(_norm_hook))

    conditions = ['B_cult', 'B_unrel']   # B_cult = match, B_unrel = mismatch
    dla_arr, S_dla = {}, {}

    try:
        # ── VALIDATION: 5 prompts before committing to the full 847x2 pass ──
        print(f"\n  [VALIDATION] 5 prompts (B_cult)")
        t0 = time.time()
        for i in range(5):
            r = dla_forward(texts_fmt['B_cult'][i])
            s1 = results_sscore['B_cult']['S'][i]
            print(f"    [{i}] dla shape={r['dla'].shape}  rms={r['rms']:7.2f}  "
                  f"sum_heads DLA={r['dla'].sum():+8.3f}  "
                  f"u.norm(resid)={r['u_dot_resid']:+8.3f}  direct(logits)={r['direct']:+8.3f}  "
                  f"S={r['S']:+7.3f}  S(stage1)={s1:+7.3f}")
            assert r['dla'].shape == (n_layers, n_heads)
            assert abs(r['u_dot_resid'] - r['direct']) < max(0.2, 0.02 * abs(r['direct'])), \
                "frozen-norm projection does not match final logits — check u / norm handling"
            assert abs(r['S'] - s1) < 1e-2, "S-score does not reproduce stage 1"
        per_prompt = (time.time() - t0) / 5
        est_min = per_prompt * (2 * n_total) / 60
        print(f"  [VALIDATION] OK — {per_prompt:.2f}s/prompt, full 847x2 pass ≈ {est_min:.1f} min")
        assert est_min < 60, f"estimated {est_min:.0f} min — too slow, aborting before full pass"

        # ── FULL PASS: 847 pairs x 2 conditions, one forward each ──
        for cond in conditions:
            print(f"\n  Computing DLA maps: {cond}")
            A  = np.zeros((n_total, n_layers, n_heads), dtype=np.float32)
            Ss = np.zeros(n_total, dtype=np.float32)
            for i, text in enumerate(texts_fmt[cond]):
                r = dla_forward(text)
                A[i] = r['dla']
                Ss[i] = r['S']
                if i % 100 == 0 and i > 0:
                    print(f"    {cond}: {i}/{n_total}")
                    torch.cuda.empty_cache()
            dla_arr[cond] = A
            S_dla[cond] = Ss
    finally:
        for h in hooks:
            h.remove()
    print("\n  Hooks removed.")

    # ── Sanity: sum over heads of DLA should correlate positively with the
    # actual S-score across prompts. It will NOT be ~1: MLPs, the direct
    # embedding path and head->head mediated effects are absent, and S uses
    # logsumexp over multiple option ids rather than the linear u readout.
    sum_dla_all = np.concatenate([dla_arr[c].sum(axis=(1, 2)) for c in conditions])
    S_all = np.concatenate([S_dla[c] for c in conditions])
    pearson_r = float(np.corrcoef(sum_dla_all, S_all)[0, 1])
    print(f"\n  Sanity: Pearson r( sum_heads DLA , S ) = {pearson_r:.3f}  (n={len(S_all)})")

    # ── Save ──
    dla_store = {
        'meta': {
            'model': ACTIVE_MODEL, 'label': CFG['label'],
            'variant': 'instruct', 'method': 'DLA (frozen final norm)',
            'reading_direction': 'u = e_c - 0.5*(e_a + e_b), unembedding rows '
                                 'averaged over option token ids',
            'option_ids': option_tokens_raw,
            'n_layers': n_layers, 'n_heads': n_heads, 'head_dim': head_dim,
            'qk_identified_heads': CFG['heads'],
        },
        'dla_match':    dla_arr['B_cult'],     # [847, n_layers, n_heads] float32
        'dla_mismatch': dla_arr['B_unrel'],
        'S_match':      S_dla['B_cult'],
        'S_mismatch':   S_dla['B_unrel'],
        'items':        list(data['items_cult']),
        'pearson_r_sumDLA_vs_S': pearson_r,
    }
    _dla_pkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_dla.pkl"
    with open(_dla_pkl, "wb") as f:
        pickle.dump(dla_store, f)
    print(f"  Saved: {_dla_pkl}  ({os.path.getsize(_dla_pkl) / 1e6:.1f} MB)")
else:
    print('has_softcapping=True — skipping the pre-norm DLA cell (the Gemma-2 branch cell runs instead)')

In [ ]:
# ── Architecture dispatch (bootstrap: DLA_DIR[bool(CFG["has_softcapping"])]) — Gemma-2 branch ──
if bool(CFG["has_softcapping"]):
    # ================================================================
    # DLA — DIRECT LOGIT ATTRIBUTION (compute), GEMMA-2 VERSION
    # Three architectural differences vs the Mistral cell:
    #  (1) Gemma RMSNorms scale by (1 + weight), not weight;
    #  (2) Gemma-2 applies post_attention_layernorm to the attention output
    #      BEFORE the residual addition -> a SECOND frozen norm per layer:
    #      contribution = g_post ⊙ (W_O[:,h-block] @ z_h) / rms(o_proj output),
    #      with rms taken from the REAL per-prompt o_proj output (captured);
    #  (3) final logits pass tanh softcapping (cap=30) -> the frozen-norm
    #      sanity check compares against PRE-CAP logits (atanh-uncapped);
    #      S-scores stay on real (capped) logits like the rest of the pipeline.
    # ================================================================
    import time

    print("=" * 80)
    print(f"DLA: DIRECT LOGIT ATTRIBUTION — {CFG['label']}")
    print("=" * 80)

    n_layers = model.config.num_hidden_layers
    n_heads  = model.config.num_attention_heads
    head_dim = getattr(model.config, "head_dim", None) or model.config.hidden_size // n_heads
    d_model  = model.config.hidden_size
    fls      = getattr(model.config, 'final_logit_softcapping', None)
    print(f"  {n_layers} layers x {n_heads} heads, head_dim={head_dim}, d_model={d_model}, "
          f"final softcap={fls}")
    print(f"  norm class: {type(model.model.norm).__name__} -> (1+w) scaling")

    def _norm_eps(m):
        return getattr(m, 'variance_epsilon', getattr(m, 'eps', 1e-6))

    # ── Reading direction (option rows averaged; 4 ids per option for Gemma) ──
    W_U = model.lm_head.weight.detach().float()
    e_opt = {opt: W_U[ids].mean(dim=0) for opt, ids in option_tokens_raw.items()}
    u = (e_opt['c'] - 0.5 * (e_opt['a'] + e_opt['b'])).to(first_device)
    del W_U, e_opt

    # ── Final frozen norm: Gemma scaling is (1 + weight) ──
    g_final  = (1.0 + model.model.norm.weight.detach().float())
    eps_final = _norm_eps(model.model.norm)
    u_eff = u * g_final

    # ── Fold u_eff and the per-layer post-attention norm gain through W_O ──
    U_proj = torch.empty(n_layers, n_heads, head_dim, device=first_device)
    eps_post = []
    for l in range(n_layers):
        lay = model.model.layers[l]
        g_post = (1.0 + lay.post_attention_layernorm.weight.detach().float())
        eps_post.append(_norm_eps(lay.post_attention_layernorm))
        W_O = lay.self_attn.o_proj.weight.detach().float()      # [d_model, n_heads*head_dim]
        U_proj[l] = (W_O.T @ (u_eff * g_post)).view(n_heads, head_dim)
        del W_O
    print(f"  U_proj precomputed: {tuple(U_proj.shape)}")

    # ── Hooks: o_proj INPUT (z), post_attention_layernorm INPUT (o_proj output,
    # for the per-layer frozen rms + decomposition check), final-norm INPUT ──
    _cap = {'z': [None] * n_layers, 'post_in': [None] * n_layers, 'resid': None}

    def _make_oproj_hook(l):
        def hook(module, args):
            _cap['z'][l] = args[0][0, -1, :].detach().float().view(n_heads, head_dim)
        return hook

    def _make_post_hook(l):
        def hook(module, args):
            _cap['post_in'][l] = args[0][0, -1, :].detach().float()
        return hook

    def _norm_hook(module, args):
        _cap['resid'] = args[0][0, -1, :].detach().float()

    def _uncap(x):
        """Invert final logit softcapping (pre-cap space)."""
        if fls is None:
            return x
        return torch.atanh(torch.clamp(x / fls, -1 + 1e-6, 1 - 1e-6)) * fls

    def dla_forward(text):
        """One forward; per-head DLA map (double frozen norm) + S + sanity pieces."""
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
        z = torch.stack(_cap['z'])                               # [L, H, hd]
        post_in = torch.stack(_cap['post_in'])                   # [L, d_model]
        resid = _cap['resid']                                    # [d_model]
        rms_post = torch.sqrt((post_in ** 2).mean(dim=1)
                              + torch.tensor(eps_post, device=post_in.device))   # [L]
        rms_final = torch.sqrt((resid ** 2).mean() + eps_final)
        dla = (torch.einsum('lhk,lhk->lh', z, U_proj)
               / rms_post[:, None] / rms_final).cpu().numpy()

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        S = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()

        pre = _uncap(logits)
        u_dot_resid = ((u_eff @ resid) / rms_final).item()
        direct = (pre[option_tokens['c']].mean()
                  - 0.5 * (pre[option_tokens['a']].mean()
                           + pre[option_tokens['b']].mean())).item()
        # per-layer decomposition check: sum over heads of W_O@z must equal the
        # captured o_proj output (post_attention_layernorm input)
        l_chk = n_layers - 1
        W_O = model.model.layers[l_chk].self_attn.o_proj.weight.detach().float()
        c_full = W_O @ z[l_chk].reshape(-1)
        rel = (c_full - post_in[l_chk]).norm().item() / (post_in[l_chk].norm().item() + 1e-9)
        del W_O
        out = {'dla': dla, 'S': S, 'rms_final': rms_final.item(),
               'u_dot_resid': u_dot_resid, 'direct': direct, 'decomp_rel_err': rel}
        del enc, outputs, logits, lp, z, post_in, resid
        return out

    hooks = []
    for l in range(n_layers):
        lay = model.model.layers[l]
        hooks.append(lay.self_attn.o_proj.register_forward_pre_hook(_make_oproj_hook(l)))
        hooks.append(lay.post_attention_layernorm.register_forward_pre_hook(_make_post_hook(l)))
    hooks.append(model.model.norm.register_forward_pre_hook(_norm_hook))

    dla_arr, S_dla = {}, {}
    try:
        print(f"\n  [VALIDATION] 5 prompts (B_cult)")
        t0 = time.time()
        for i in range(5):
            r = dla_forward(texts_fmt['B_cult'][i])
            s1 = results_sscore['B_cult']['S'][i]
            print(f"    [{i}] dla shape={r['dla'].shape}  sum_heads DLA={r['dla'].sum():+8.3f}  "
                  f"u.norm(resid)={r['u_dot_resid']:+8.3f}  direct(pre-cap)={r['direct']:+8.3f}  "
                  f"decomp_err={r['decomp_rel_err']:.4f}  S={r['S']:+7.3f}  S(stage1)={s1:+7.3f}")
            assert r['dla'].shape == (n_layers, n_heads)
            assert r['decomp_rel_err'] < 0.02, "per-head o_proj decomposition broken"
            assert abs(r['u_dot_resid'] - r['direct']) < max(0.5, 0.05 * abs(r['direct'])), \
                "frozen-norm projection does not match pre-cap logits"
            assert abs(r['S'] - s1) < 1e-2, "S-score does not reproduce stage 1"
        per_prompt = (time.time() - t0) / 5
        est_min = per_prompt * (2 * n_total) / 60
        print(f"  [VALIDATION] OK — {per_prompt:.2f}s/prompt, full 847x2 ≈ {est_min:.1f} min")
        assert est_min < 60, f"estimated {est_min:.0f} min — too slow"

        for cond in conditions:
            print(f"\n  Computing DLA maps: {cond}")
            A  = np.zeros((n_total, n_layers, n_heads), dtype=np.float32)
            Ss = np.zeros(n_total, dtype=np.float32)
            for i, text in enumerate(texts_fmt[cond]):
                r = dla_forward(text)
                A[i] = r['dla']
                Ss[i] = r['S']
                if i % 200 == 0 and i > 0:
                    print(f"    {cond}: {i}/{n_total}")
                    torch.cuda.empty_cache()
            dla_arr[cond] = A
            S_dla[cond] = Ss
    finally:
        for h in hooks:
            h.remove()
    hooks = []
    print("\n  Hooks removed.")

    sum_dla_all = np.concatenate([dla_arr[c].sum(axis=(1, 2)) for c in conditions])
    S_all = np.concatenate([S_dla[c] for c in conditions])
    pearson_r = float(np.corrcoef(sum_dla_all, S_all)[0, 1])
    print(f"\n  Sanity: Pearson r( sum_heads DLA , S ) = {pearson_r:.3f}  (n={len(S_all)})")

    dla_store = {
        'meta': {
            'model': ACTIVE_MODEL, 'label': CFG['label'], 'variant': 'instruct',
            'method': 'DLA (frozen post-attention norm per layer + frozen final '
                      'norm, both (1+w); sanity check in pre-softcap logit space)',
            'option_ids': option_tokens_raw,
            'n_layers': n_layers, 'n_heads': n_heads, 'head_dim': head_dim,
            'qk_identified_heads': CFG['heads'],
        },
        'dla_match':    dla_arr['B_cult'],
        'dla_mismatch': dla_arr['B_unrel'],
        'S_match':      S_dla['B_cult'],
        'S_mismatch':   S_dla['B_unrel'],
        'items':        list(data['items_cult']),
        'pearson_r_sumDLA_vs_S': pearson_r,
    }
    _dla_pkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_dla.pkl"
    with open(_dla_pkl, "wb") as f:
        pickle.dump(dla_store, f)
    print(f"  Saved: {_dla_pkl}  ({os.path.getsize(_dla_pkl) / 1e6:.1f} MB)")
else:
    print('has_softcapping=False — skipping the Gemma-2 DLA cell (the pre-norm branch cell already ran)')

## Directional DLA (Test 4) — architecture dispatch

The directional DLA cells (39 pre-norm / 34 Gemma-2); exactly
one runs, selected by `CFG["has_softcapping"]`. Provides `u_dir_eff`, `sign`
and `eps_final` (plus, on the Gemma-2 branch, the `(1+g)` per-layer norm
folds) consumed by the neuron ranking cell below.

In [ ]:
# ── Bootstrap glue: the directional
# cells below need items_arr at their tail. ──
items_arr = np.array(data['items_cult'])
# ttest_1samp / ttest_rel already imported above

In [ ]:
# ── Architecture dispatch (bootstrap: DLA_DIR[bool(CFG["has_softcapping"])]) — pre-norm branch ──
if not bool(CFG["has_softcapping"]):
    # ================================================================
    # TEST 4-M — DIRECTIONAL LENS + DIRECTIONAL DLA (MISTRAL)
    # Same logic as the Gemma directional pass, Mistral architecture:
    # plain pre-norm residual (attn and MLP outputs add directly), final
    # RMSNorm scales by w (not 1+w), no logit softcapping.
    # ================================================================
    print("=" * 80)
    print("TEST 4-M (MISTRAL) — DIRECTIONAL LENS + DLA_dir")
    print("=" * 80)

    W_U = model.lm_head.weight.detach().float()
    e_a = W_U[option_tokens_raw['a']].mean(dim=0)
    e_b = W_U[option_tokens_raw['b']].mean(dim=0)
    u_ab = (e_a - e_b).to(first_device)
    del W_U, e_a, e_b
    norm_w = model.model.norm.weight.detach().float()
    eps_final = getattr(model.model.norm, 'variance_epsilon',
                        getattr(model.model.norm, 'eps', 1e-6))
    u_dir_eff = u_ab * norm_w

    U_dir = torch.empty(n_layers, n_heads, head_dim, device=first_device)
    for l in range(n_layers):
        W_O = model.model.layers[l].self_attn.o_proj.weight.detach().float()
        U_dir[l] = (W_O.T @ u_dir_eff).view(n_heads, head_dim)
        del W_O

    _c4 = {'z': [None]*n_layers, 'mlp_out': [None]*n_layers,
           'lay_out': [None]*n_layers, 'resid': None}
    def _h_z(l):
        def hook(m, a): _c4['z'][l] = a[0][0, -1, :].detach().float().view(n_heads, head_dim)
        return hook
    def _h_mlp(l):
        def hook(m, a, out):
            _c4['mlp_out'][l] = (out[0] if isinstance(out, tuple) else out)[0, -1, :].detach().float()
        return hook
    def _h_out(l):
        def hook(m, a, out):
            _c4['lay_out'][l] = (out[0] if isinstance(out, tuple) else out)[0, -1, :].detach().float()
        return hook
    def _h_resid(m, a): _c4['resid'] = a[0][0, -1, :].detach().float()

    def dir_forward(text):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            o = model(**enc)
        z = torch.stack(_c4['z'])
        mlp_out = torch.stack(_c4['mlp_out'])
        lay_out = torch.stack(_c4['lay_out'])
        resid = _c4['resid']
        rms_fin = torch.sqrt((resid ** 2).mean() + eps_final)
        dla_dir = (torch.einsum('lhk,lhk->lh', z, U_dir) / rms_fin)
        mlp_dir = ((mlp_out @ u_dir_eff) / rms_fin)
        rms_lay = torch.sqrt((lay_out ** 2).mean(1) + eps_final)
        cum_dir = ((lay_out @ u_dir_eff) / rms_lay)
        logits = o.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lps = [torch.logsumexp(lp[option_tokens[k]], dim=0).item() for k in ['a', 'b', 'c']]
        direct_ab = (logits[option_tokens['a']].mean() - logits[option_tokens['b']].mean()).item()
        u_fin = ((u_dir_eff @ resid) / rms_fin).item()
        out = {'dla_dir': dla_dir.cpu().numpy(), 'mlp_dir': mlp_dir.cpu().numpy(),
               'cum_dir': cum_dir.cpu().numpy(), 'lps': lps,
               'u_fin': u_fin, 'direct_ab': direct_ab}
        del enc, o, z, mlp_out, lay_out, resid, logits, lp
        return out

    hooks = []
    for l in range(n_layers):
        lay_l = model.model.layers[l]
        hooks.append(lay_l.self_attn.o_proj.register_forward_pre_hook(_h_z(l)))
        hooks.append(lay_l.mlp.register_forward_hook(_h_mlp(l)))
        hooks.append(lay_l.register_forward_hook(_h_out(l)))
    del lay_l
    hooks.append(model.model.norm.register_forward_pre_hook(_h_resid))

    sign = np.where(np.array(data['assoc_pos']) == 'a', 1.0, -1.0)
    res4 = {}
    try:
        print("\n  [VALIDATION] 5 prompts (B_cult)")
        t0 = time.time()
        for i in range(5):
            r = dir_forward(texts_fmt['B_cult'][i])
            print(f"    [{i}] u.norm(resid)={r['u_fin']:+8.3f}  direct(a-b)={r['direct_ab']:+8.3f}  "
                  f"cum[last]={r['cum_dir'][-1]:+8.3f}")
            assert abs(r['u_fin'] - r['direct_ab']) < max(0.3, 0.05 * abs(r['direct_ab']))
        est = (time.time() - t0) / 5 * 2 * n_total / 60
        print(f"  [VALIDATION] OK — full pass ≈ {est:.1f} min")
        assert est < 60

        for cond in conditions:
            print(f"\n  directional pass: {cond}")
            res4[cond] = {'dla_dir': np.zeros((n_total, n_layers, n_heads), np.float32),
                          'mlp_dir': np.zeros((n_total, n_layers), np.float32),
                          'cum_dir': np.zeros((n_total, n_layers), np.float32),
                          'lps': np.zeros((n_total, 3), np.float64)}
            for i, text in enumerate(texts_fmt[cond]):
                r = dir_forward(text)
                res4[cond]['dla_dir'][i] = r['dla_dir']
                res4[cond]['mlp_dir'][i] = r['mlp_dir']
                res4[cond]['cum_dir'][i] = r['cum_dir']
                res4[cond]['lps'][i] = r['lps']
                if i % 300 == 0 and i > 0:
                    print(f"    {cond}: {i}/{n_total}")
                    torch.cuda.empty_cache()
    finally:
        for h in hooks:
            h.remove()
        hooks = []
    print("  hooks removed")

    def m_sgn(cond, key):
        v = res4[cond][key]
        return (v * sign.reshape(-1, *([1] * (v.ndim - 1)))).mean(axis=0)

    d_cum  = m_sgn('B_cult', 'cum_dir') - m_sgn('B_unrel', 'cum_dir')
    d_mlp  = m_sgn('B_cult', 'mlp_dir') - m_sgn('B_unrel', 'mlp_dir')
    d_head = m_sgn('B_cult', 'dla_dir') - m_sgn('B_unrel', 'dla_dir')
    d_attn = d_head.sum(axis=1)

    def pR_of(cond):
        lps = res4[cond]['lps']
        lp_R = np.where(sign > 0, lps[:, 0], lps[:, 1])
        lp_U = np.where(sign > 0, lps[:, 1], lps[:, 0])
        return 1 / (1 + np.exp(-(lp_R - lp_U)))
    print(f"\n  P(R) match = {pR_of('B_cult').mean():.4f}  |  mismatch = {pR_of('B_unrel').mean():.4f}")

    print(f"\n  DIRECTIONAL CONTRAST BY LAYER (every 2nd layer; Δ = match − mismatch)")
    print(f"  {'layer':>5s}  {'Δcum':>8s}  {'Δattn':>8s}  {'ΔMLP':>8s}")
    for l in range(0, n_layers, 2):
        print(f"  {l:5d}  {d_cum[l]:+8.4f}  {d_attn[l]:+8.4f}  {d_mlp[l]:+8.4f}  "
              f"{'#' * int(min(40, abs(d_cum[l]) * 20))}")
    print(f"  {n_layers-1:5d}  {d_cum[-1]:+8.4f}  {d_attn[-1]:+8.4f}  {d_mlp[-1]:+8.4f}")
    print(f"  totals: Σ Δattn = {d_attn.sum():+.4f}   Σ ΔMLP = {d_mlp.sum():+.4f}   "
          f"final Δcum = {d_cum[-1]:+.4f}")

    flat_dir = [(l, h, d_head[l, h]) for l in range(n_layers) for h in range(n_heads)]
    flat_dir.sort(key=lambda x: -abs(x[2]))
    items_u = np.unique(items_arr)
    sdla_m = res4['B_cult']['dla_dir'] * sign[:, None, None]
    sdla_u = res4['B_unrel']['dla_dir'] * sign[:, None, None]
    print(f"\n  TOP 15 HEADS BY |ΔDLA_dir|")
    for rank, (l, h, d) in enumerate(flat_dir[:15], 1):
        di = np.array([(sdla_m[items_arr == it, l, h].mean()
                        - sdla_u[items_arr == it, l, h].mean()) for it in items_u])
        t_lh, p_lh = ttest_1samp(di, 0)
        mk = "  <== old format-DLA candidate" if (l, h) in [(31, 22), (29, 2)] else \
             ("  <-- QK-identified" if (l, h) in [(8, 16), (9, 23), (12, 9)] else "")
        print(f"  {rank:4d}  L{l:<2d}H{h:<3d}  {d:+9.4f}  t={t_lh:+6.2f}  p={p_lh:.1e}{mk}")
    rank_dir = {(l, h): r for r, (l, h, _) in enumerate(flat_dir, 1)}
    print(f"\n  directional ranks: L31H22 -> {rank_dir[(31, 22)]}, L29H2 -> {rank_dir[(29, 2)]}, "
          f"QK: L8H16 -> {rank_dir[(8, 16)]}, L9H23 -> {rank_dir[(9, 23)]}, L12H9 -> {rank_dir[(12, 9)]}")
else:
    print('has_softcapping=True — skipping the pre-norm directional cell (the Gemma-2 branch cell runs instead)')

In [ ]:
# ── Architecture dispatch (bootstrap: DLA_DIR[bool(CFG["has_softcapping"])]) — Gemma-2 branch ──
if bool(CFG["has_softcapping"]):
    # ================================================================
    # TEST 4 — DIRECTIONAL REANALYSIS (the binding direction, not the format
    # margin). Reading direction u_ab = e_a - e_b, signed per prompt by
    # assoc_pos (R in (a) -> +1, R in (b) -> -1), so DLA_dir > 0 = "writes
    # toward the R position". One pass over 847x2 captures simultaneously:
    #   (i)  directional logit lens: cumulative residual projection per layer
    #        (where in depth does the match/mismatch contrast appear?);
    #   (ii) attention-block vs MLP-block directional contribution per layer
    #        (WHO writes it — heads or MLPs?);
    #   (iii) per-head directional DLA (which heads, if any).
    # All frozen-norm conventions as in the Gemma DLA cell ((1+w), real rms).
    # ================================================================
    import time

    print("=" * 80)
    print("TEST 4 (GEMMA) — DIRECTIONAL LENS + DIRECTIONAL DLA")
    print("=" * 80)

    # ── Directional reading vector ──
    W_U = model.lm_head.weight.detach().float()
    e_a = W_U[option_tokens_raw['a']].mean(dim=0)
    e_b = W_U[option_tokens_raw['b']].mean(dim=0)
    u_ab = (e_a - e_b).to(first_device)
    del W_U, e_a, e_b
    u_dir_eff = u_ab * g_final                                  # fold final (1+w)

    # per-layer folds
    U_dir = torch.empty(n_layers, n_heads, head_dim, device=first_device)
    V_mlp = torch.empty(n_layers, d_model, device=first_device)  # u folded with post-FF gain
    for l in range(n_layers):
        lay = model.model.layers[l]
        g_post = (1.0 + lay.post_attention_layernorm.weight.detach().float())
        g_pf   = (1.0 + lay.post_feedforward_layernorm.weight.detach().float())
        W_O = lay.self_attn.o_proj.weight.detach().float()
        U_dir[l] = (W_O.T @ (u_dir_eff * g_post)).view(n_heads, head_dim)
        V_mlp[l] = u_dir_eff * g_pf
        del W_O
    eps_pf = [_norm_eps(model.model.layers[l].post_feedforward_layernorm)
              for l in range(n_layers)]

    # ── Hooks: z (o_proj in), post-attn in, post-FF in, layer outputs, final norm in ──
    _c4 = {'z': [None]*n_layers, 'attn_in': [None]*n_layers,
           'mlp_in': [None]*n_layers, 'lay_out': [None]*n_layers, 'resid': None}
    def _h_z(l):
        def hook(m, a): _c4['z'][l] = a[0][0, -1, :].detach().float().view(n_heads, head_dim)
        return hook
    def _h_attn(l):
        def hook(m, a): _c4['attn_in'][l] = a[0][0, -1, :].detach().float()
        return hook
    def _h_mlp(l):
        def hook(m, a): _c4['mlp_in'][l] = a[0][0, -1, :].detach().float()
        return hook
    def _h_out(l):
        def hook(m, a, out):
            _c4['lay_out'][l] = (out[0] if isinstance(out, tuple) else out)[0, -1, :].detach().float()
        return hook
    def _h_resid(m, a): _c4['resid'] = a[0][0, -1, :].detach().float()

    def dir_forward(text):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad():
            o = model(**enc)
        z = torch.stack(_c4['z'])
        attn_in = torch.stack(_c4['attn_in'])      # [L, d]
        mlp_in  = torch.stack(_c4['mlp_in'])       # [L, d]
        lay_out = torch.stack(_c4['lay_out'])      # [L, d]
        resid   = _c4['resid']
        rms_post = torch.sqrt((attn_in ** 2).mean(1) + torch.tensor(eps_post, device=attn_in.device))
        rms_pf   = torch.sqrt((mlp_in ** 2).mean(1) + torch.tensor(eps_pf, device=mlp_in.device))
        rms_fin  = torch.sqrt((resid ** 2).mean() + eps_final)
        dla_dir  = (torch.einsum('lhk,lhk->lh', z, U_dir) / rms_post[:, None] / rms_fin)
        mlp_dir  = ((V_mlp * mlp_in).sum(1) / rms_pf / rms_fin)
        rms_lay  = torch.sqrt((lay_out ** 2).mean(1) + eps_final)
        cum_dir  = ((lay_out @ u_dir_eff) / rms_lay)           # lens with own-rms final norm
        logits = o.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lps = [torch.logsumexp(lp[option_tokens[k]], dim=0).item() for k in ['a', 'b', 'c']]
        pre = _uncap(logits)
        direct_ab = (pre[option_tokens['a']].mean() - pre[option_tokens['b']].mean()).item()
        u_fin = ((u_dir_eff @ resid) / rms_fin).item()
        out = {'dla_dir': dla_dir.cpu().numpy(), 'mlp_dir': mlp_dir.cpu().numpy(),
               'cum_dir': cum_dir.cpu().numpy(), 'lps': lps,
               'u_fin': u_fin, 'direct_ab': direct_ab}
        del enc, o, z, attn_in, mlp_in, lay_out, resid, logits, lp, pre
        return out

    hooks = []
    for l in range(n_layers):
        lay = model.model.layers[l]
        hooks.append(lay.self_attn.o_proj.register_forward_pre_hook(_h_z(l)))
        hooks.append(lay.post_attention_layernorm.register_forward_pre_hook(_h_attn(l)))
        hooks.append(lay.post_feedforward_layernorm.register_forward_pre_hook(_h_mlp(l)))
        hooks.append(lay.register_forward_hook(_h_out(l)))
    hooks.append(model.model.norm.register_forward_pre_hook(_h_resid))

    sign = np.where(np.array(data['assoc_pos']) == 'a', 1.0, -1.0)
    res4 = {}
    try:
        print("\n  [VALIDATION] 5 prompts (B_cult)")
        t0 = time.time()
        for i in range(5):
            r = dir_forward(texts_fmt['B_cult'][i])
            print(f"    [{i}] u.norm(resid)={r['u_fin']:+8.3f}  direct(pre-cap a-b)={r['direct_ab']:+8.3f}  "
                  f"cum[last]={r['cum_dir'][-1]:+8.3f}  sum dla_dir={r['dla_dir'].sum():+7.3f}  "
                  f"sum mlp_dir={r['mlp_dir'].sum():+7.3f}")
            assert abs(r['u_fin'] - r['direct_ab']) < max(0.5, 0.05 * abs(r['direct_ab'])), \
                "directional frozen-norm projection broken"
        est = (time.time() - t0) / 5 * 2 * n_total / 60
        print(f"  [VALIDATION] OK — full pass ≈ {est:.1f} min")
        assert est < 60

        for cond in conditions:
            print(f"\n  directional pass: {cond}")
            n = n_total
            res4[cond] = {'dla_dir': np.zeros((n, n_layers, n_heads), np.float32),
                          'mlp_dir': np.zeros((n, n_layers), np.float32),
                          'cum_dir': np.zeros((n, n_layers), np.float32),
                          'lps': np.zeros((n, 3), np.float64)}
            for i, text in enumerate(texts_fmt[cond]):
                r = dir_forward(text)
                res4[cond]['dla_dir'][i] = r['dla_dir']
                res4[cond]['mlp_dir'][i] = r['mlp_dir']
                res4[cond]['cum_dir'][i] = r['cum_dir']
                res4[cond]['lps'][i] = r['lps']
                if i % 300 == 0 and i > 0:
                    print(f"    {cond}: {i}/{n_total}")
                    torch.cuda.empty_cache()
    finally:
        for h in hooks:
            h.remove()
        hooks = []
    print("  hooks removed")

    # ── Aggregate: toward-R values = sign * (a-b) projections ──
    def m(cond, key, axis0=True):
        v = res4[cond][key]
        return (v * sign.reshape(-1, *([1] * (v.ndim - 1)))).mean(axis=0)

    d_cum  = m('B_cult', 'cum_dir')  - m('B_unrel', 'cum_dir')    # [L]
    d_mlp  = m('B_cult', 'mlp_dir')  - m('B_unrel', 'mlp_dir')    # [L]
    d_head = m('B_cult', 'dla_dir')  - m('B_unrel', 'dla_dir')    # [L, H]
    d_attn = d_head.sum(axis=1)                                   # [L]

    # behavioral anchor: P(R) match vs mismatch from the same pass
    def pR(cond):
        lps = res4[cond]['lps']
        lp_R = np.where(sign > 0, lps[:, 0], lps[:, 1])
        lp_U = np.where(sign > 0, lps[:, 1], lps[:, 0])
        return 1 / (1 + np.exp(-(lp_R - lp_U)))
    print(f"\n  P(R) match = {pR('B_cult').mean():.4f} (test2 baseline: 0.5988)  |  "
          f"P(R) mismatch = {pR('B_unrel').mean():.4f} (expect ~0.5)")

    print(f"\n  DIRECTIONAL CONTRAST BY LAYER (Δ = match − mismatch, toward-R units)")
    print(f"  {'layer':>5s}  {'Δcum (lens)':>11s}  {'Δattn-block':>11s}  {'ΔMLP-block':>10s}")
    for l in range(n_layers):
        bar = '#' * int(min(40, abs(d_cum[l]) * 20))
        print(f"  {l:5d}  {d_cum[l]:+11.4f}  {d_attn[l]:+11.4f}  {d_mlp[l]:+10.4f}  {bar}")
    print(f"\n  totals: Σ Δattn = {d_attn.sum():+.4f}   Σ ΔMLP = {d_mlp.sum():+.4f}   "
          f"final Δcum = {d_cum[-1]:+.4f}")

    # ── Top directional heads ──
    flat_dir = [(l, h, d_head[l, h]) for l in range(n_layers) for h in range(n_heads)]
    flat_dir.sort(key=lambda x: -abs(x[2]))
    items_u = np.unique(items_arr)
    sdla_m = res4['B_cult']['dla_dir'] * sign[:, None, None]
    sdla_u = res4['B_unrel']['dla_dir'] * sign[:, None, None]
    print(f"\n  TOP 15 HEADS BY |ΔDLA_dir| (toward-R direct path)")
    print(f"  {'rank':>4s}  {'head':>8s}  {'ΔDLA_dir':>9s}  {'t(66)':>7s}  {'p':>10s}")
    for rank, (l, h, d) in enumerate(flat_dir[:15], 1):
        di = np.array([(sdla_m[items_arr == it, l, h].mean()
                        - sdla_u[items_arr == it, l, h].mean()) for it in items_u])
        t_lh, p_lh = ttest_1samp(di, 0)
        qk = "  <-- QK-identified" if (l, h) in [(11, 14), (13, 13), (16, 15)] else ""
        print(f"  {rank:4d}  L{l:<2d}H{h:<3d}  {d:+9.4f}  {t_lh:+7.2f}  {p_lh:10.2e}{qk}")
    rank_dir = {(l, h): r for r, (l, h, _) in enumerate(flat_dir, 1)}
    print(f"\n  QK heads in the DIRECTIONAL ranking: "
          f"L11H14 -> {rank_dir[(11, 14)]}, L13H13 -> {rank_dir[(13, 13)]}, "
          f"L16H15 -> {rank_dir[(16, 15)]}  (of {n_layers * n_heads})")
else:
    print('has_softcapping=False — skipping the Gemma-2 directional cell (the pre-norm branch cell already ran)')

In [ ]:
# ── Bootstrap glue: scoring helpers
# validated against stage 1, neutral-prompt subsample (900, seed 7), and the
# variables-present gate for the ranking/ablation cells. ──
# helpers (validated to reproduce stage 1)
def compute_lps_unbatched(texts, progress_every=None, label=""):
    out = np.zeros((len(texts), 3), dtype=np.float64)
    for i, text in enumerate(texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        with torch.no_grad(): o = model(**enc)
        lp = F.log_softmax(o.logits[0,-1,:].float(), dim=-1)
        out[i,0]=torch.logsumexp(lp[option_tokens['a']],0).item()
        out[i,1]=torch.logsumexp(lp[option_tokens['b']],0).item()
        out[i,2]=torch.logsumexp(lp[option_tokens['c']],0).item()
        del enc,o,lp
        if progress_every and i%progress_every==0 and i>0:
            print(f"    {label}: {i}/{len(texts)}"); torch.cuda.empty_cache()
    return out
def S_from_lps(lps): return lps[:,2] - np.logaddexp(lps[:,0], lps[:,1])

# neutral prompts (subsample 900, seed 7 — same as Test 3)
neutral_q = [it[0] for it in neutral_items if extract_options(it[0])[0] is not None]
_alln = format_for_chat(neutral_q, tokenizer)
_r = np.random.RandomState(7)
_sel = _r.choice(len(_alln), size=min(900, len(_alln)), replace=False)
texts_neutral = [_alln[i] for i in sorted(_sel)]

_chk = np.abs(S_from_lps(compute_lps_unbatched(texts_fmt['B_cult'][:5]))
              - results_sscore['B_cult']['S'][:5]).max()
print(f"\n  helper drift vs stage1 = {_chk:.2e}")
assert _chk < 1e-4
_need=["u_dir_eff","sign","n_layers","eps_final","items_arr","conditions","texts_fmt",
       "results_sscore","compute_lps_unbatched","S_from_lps","texts_neutral",
       "option_tokens","first_device","CFG","OUTPUT_DIR","ACTIVE_MODEL"]
_miss=[n for n in _need if n not in globals()]
assert not _miss, f"missing: {_miss}"
print("\n  Bootstrap OK —", ACTIVE_MODEL, "| neutral:", len(texts_neutral))

## Cell 1 — neuron-DLA + specificity-ratio ranking (neutral split A)

`TOPN_POOL = 300` top-`|Δdir|` neurons, re-ranked by the continuous ratio
`|Δdir| / (leak + eps)`, `RANK_KEEP = 200` kept. Saves the ranking to
`results_<MODEL_KEY>_instruct_mlp_neuron_filtered.pkl`.

In [ ]:
# ================================================================
# MLP NEURON-DLA  +  SPECIFICITY RATIO RANKING  (split A)
# Run AFTER the directional Test-4 cell (provides u_dir_eff, sign,
# n_layers, eps_final, items_arr, conditions, texts_fmt, texts_neutral).
#
# Instead of a hard filter (rank by |Δdir| then drop high-leak), we rank by a
# CONTINUOUS specificity score  spec = |Δdir| / (leak + eps).  This lets a
# strong-but-slightly-leaky neuron outrank a weak-but-clean one, neuron by
# neuron, instead of a 50%-cutoff guillotine that destroyed Mistral's effect.
#   Δdir : directional contribution (toward-R) on CULTURAL prompts
#   leak : RMS a-vs-b contribution on NEUTRAL split-A prompts
# The independent S_neutral control (split B) still lives in the ablation cell.
# ================================================================
import time
from scipy.stats import ttest_1samp

assert 'u_dir_eff' in globals() and 'sign' in globals(), "run the directional cell first"
print("=" * 80)
print(f"MLP NEURON-DLA + SPEC-RATIO RANKING — {CFG['label']}")
print("=" * 80)

d_ff = model.config.intermediate_size
SOFTCAP = bool(CFG.get('has_softcapping', False))
LEAK_EPS_FRAC = 0.10        # eps = LEAK_EPS_FRAC * median(leak over the top-|Δdir| pool)
TOPN_POOL     = 300         # consider this many top-|Δdir| neurons before ratio re-ranking
RANK_KEEP     = 200         # keep this many after ratio ranking (feeds the K-sweep)

# ── neutral split A (selection) / B (control), seed 7 ──
_rngs = np.random.RandomState(7)
_perm = _rngs.permutation(len(texts_neutral))
_half = len(texts_neutral) // 2
neutralA = [texts_neutral[i] for i in sorted(_perm[:_half])]
neutralB = [texts_neutral[i] for i in sorted(_perm[_half:])]
print(f"  neutral split: A(filter)={len(neutralA)}  B(control)={len(neutralB)}  seed=7")

# ── folded down-projection onto the a-vs-b letter direction ──
w_down_u = torch.empty(n_layers, d_ff, device=first_device)
g_ff, eps_ff = ([], []) if SOFTCAP else (None, None)
for l in range(n_layers):
    lay = model.model.layers[l]
    W_down = lay.mlp.down_proj.weight.detach().float()
    if SOFTCAP:
        g = (1.0 + lay.post_feedforward_layernorm.weight.detach().float())
        eps_ff.append(getattr(lay.post_feedforward_layernorm, 'variance_epsilon',
                              getattr(lay.post_feedforward_layernorm, 'eps', 1e-6)))
        g_ff.append(g); u_l = u_dir_eff * g
    else:
        u_l = u_dir_eff
    w_down_u[l] = W_down.T @ u_l
    del W_down
print(f"  w_down_u: {tuple(w_down_u.shape)}  softcap={SOFTCAP}")

_cn = {'a': [None]*n_layers, 'mlp_out': [None]*n_layers, 'resid': None}
def _h_a(l):
    def hook(m, args): _cn['a'][l] = args[0][0,-1,:].detach().float()
    return hook
def _h_mo(l):
    def hook(m, args, out):
        _cn['mlp_out'][l] = (out[0] if isinstance(out, tuple) else out)[0,-1,:].detach().float()
    return hook
def _h_rf(m, args): _cn['resid'] = args[0][0,-1,:].detach().float()

def neuron_contrib(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(first_device) for k, v in enc.items()}
    with torch.no_grad(): _ = model(**enc)
    A = torch.stack(_cn['a'])
    if SOFTCAP:
        MO = torch.stack(_cn['mlp_out'])
        rms = torch.sqrt((MO**2).mean(1) + torch.tensor(eps_ff, device=first_device))
    else:
        rms = torch.sqrt((_cn['resid']**2).mean() + eps_final).expand(n_layers)
    c = (A * w_down_u) / rms[:, None]
    del enc, _
    return c.cpu().numpy()

hooks = []
for l in range(n_layers):
    lay = model.model.layers[l]
    hooks.append(lay.mlp.down_proj.register_forward_pre_hook(_h_a(l)))
    hooks.append(lay.mlp.register_forward_hook(_h_mo(l)))
hooks.append(model.model.norm.register_forward_pre_hook(_h_rf))

try:
    print("\n  [VALIDATION] neuron-sum == block (5 prompts)")
    for i in range(5):
        c = neuron_contrib(texts_fmt['B_cult'][i])
        if SOFTCAP:
            MO = torch.stack(_cn['mlp_out'])
            rms_b = torch.sqrt((MO**2).mean(1) + torch.tensor(eps_ff, device=first_device))
            block = (((MO*torch.stack(g_ff)) @ u_dir_eff)/rms_b).cpu().numpy()
        else:
            MO = torch.stack(_cn['mlp_out']); rms_b = torch.sqrt((_cn['resid']**2).mean()+eps_final)
            block = ((MO @ u_dir_eff)/rms_b).cpu().numpy()
        ml = int(np.argmax(np.abs(block)))
        assert abs(c.sum(1)[ml]-block[ml]) < max(1e-2, 5e-3*abs(block[ml]))
    print("  [VALIDATION] OK\n")

    C = {}
    for cond in conditions:
        print(f"  cultural pass: {cond}")
        arr = np.zeros((len(texts_fmt[cond]), n_layers, d_ff), dtype=np.float32)
        for i, t in enumerate(texts_fmt[cond]):
            arr[i] = neuron_contrib(t)
            if i % 300 == 0 and i > 0: torch.cuda.empty_cache()
        C[cond] = arr

    print(f"  neutral-A leak pass ({len(neutralA)} prompts)")
    sumsq = np.zeros((n_layers, d_ff), dtype=np.float64)
    for i, t in enumerate(neutralA):
        c = neuron_contrib(t)
        sumsq += c.astype(np.float64)**2
        if i % 300 == 0 and i > 0: torch.cuda.empty_cache()
    leak = np.sqrt(sumsq / len(neutralA)).astype(np.float32)
finally:
    for h in hooks: h.remove()
print("  hooks removed")

# ── signed item-level Δdir on cultural ──
sm = C['B_cult'] * sign[:, None, None]
su = C['B_unrel'] * sign[:, None, None]
items_u = np.unique(items_arr)
im = lambda x: np.stack([x[items_arr==it].mean(0) for it in items_u])
delta_dir = (im(sm) - im(su)).mean(0)                            # [L, d_ff]

# ── RATIO RANKING ──
# take the top-|Δdir| pool, set eps from its leak scale, then rank by spec ratio
flat = sorted(((l, j) for l in range(n_layers) for j in range(d_ff)),
              key=lambda lj: -abs(delta_dir[lj[0], lj[1]]))[:TOPN_POOL]
leak_pool = np.array([leak[l, j] for (l, j) in flat])
eps = LEAK_EPS_FRAC * np.median(leak_pool)                       # stabilises tiny-leak blow-ups
scored = [(l, j, float(delta_dir[l, j]), float(leak[l, j]),
           abs(delta_dir[l, j]) / (leak[l, j] + eps)) for (l, j) in flat]
scored.sort(key=lambda x: -x[4])                                 # by spec ratio, descending
kept = scored[:RANK_KEEP]
print(f"\n  ratio ranking: pool={TOPN_POOL}, eps={eps:.4f}, keep top-{RANK_KEEP} by |Δdir|/(leak+eps)")

print(f"\n  TOP 30 BY SPEC RATIO")
print(f"  {'rank':>4} {'neuron':>10} {'Δdir':>9} {'leak':>8} {'ratio':>8}")
for r, (l, j, d, lk, s) in enumerate(kept[:30], 1):
    print(f"  {r:4d} L{l:<2d}N{j:<6d} {d:+9.4f} {lk:8.4f} {s:8.2f}")

n_pos = sum(1 for (l,j,d,lk,s) in kept if d > 0)
n_neg = sum(1 for (l,j,d,lk,s) in kept if d < 0)
print(f"\n  kept: {n_pos} toward-R, {n_neg} anti-R")

neuron_store = {
    'meta': {'model': ACTIVE_MODEL, 'label': CFG['label'], 'softcap': SOFTCAP,
             'TOPN_POOL': TOPN_POOL, 'RANK_KEEP': RANK_KEEP, 'eps': float(eps),
             'rank_by': 'spec_ratio |Δdir|/(leak+eps)'},
    'kept': [(l, j, float(d), float(lk)) for (l, j, d, lk, s) in kept],   # ranked by ratio
    'neutralB': neutralB,
    'delta_dir': delta_dir.astype(np.float32), 'leak': leak,
}
_npkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_mlp_neuron_filtered.pkl"
with open(_npkl, "wb") as f: pickle.dump(neuron_store, f)
print(f"\n  Saved: {_npkl}")
print("  -> feed neuron_store['kept'] into the ablation+sweep cell (unchanged).")

## Cell 2 — K-sweep {10, 20, 40} + split-B neutral control

Mean-ablation of the top-K POS / NEG neurons (`K_SWEEP = [10, 20, 40]`);
ΔP(R) on the 847 cultural pairs and S-shift on neutral split B. Updates the
`_mlp_neuron_filtered.pkl` pickle in place with `sweep` and `baseline`.

In [ ]:
# ================================================================
# MLP NEURON ABLATION — K-SWEEP {10,20,40,80}  +  SPLIT-B CONTROL
# Run AFTER cell1_filter (uses neuron_store['kept'], neuron_store['neutralB']).
# Selection already specificity-filtered on neutral split A; the S_neutral
# control here uses split B -> independent of selection (non-circular).
#
#   POS = toward-R kept neurons (Δdir>0), NEG = anti-R kept (Δdir<0)
#   For each K: ablate top-K of each sign, measure
#     ΔP(R) on cultural match (847)  and  S_neutral shift on split B.
#   Clean directional component: POS ΔP(R)<0, NEG ΔP(R)>0, small neutral shift.
# ================================================================
import time
from scipy.stats import ttest_1samp

assert 'neuron_store' in globals() and 'kept' in neuron_store, "run cell1_filter first"
K_SWEEP = [10, 20, 40]
neutralB = neuron_store['neutralB']

kept = neuron_store['kept']                      # (l, j, Δdir, leak), ranked by |Δdir|
pos_rank = [(l, j) for (l, j, d, lk) in kept if d > 0]
neg_rank = [(l, j) for (l, j, d, lk) in kept if d < 0]
print(f"  kept pool: {len(pos_rank)} toward-R, {len(neg_rank)} anti-R")

def to_dict(pairs):
    out = {}
    for l, j in pairs: out.setdefault(l, []).append(j)
    return out

# ── global mean neuron activations (50+50 baseline) ──
if 'neuron_mean_acts' not in globals():
    print("  computing mean neuron activations...")
    d_ff = model.config.intermediate_size
    _sum = torch.zeros(n_layers, d_ff, dtype=torch.float64, device=first_device); _nt = 0
    def _mk(l):
        def hook(m, args): _sum[l] += args[0][0].detach().to(torch.float64).sum(0)
        return hook
    _mh = [model.model.layers[l].mlp.down_proj.register_forward_pre_hook(_mk(l)) for l in range(n_layers)]
    try:
        with torch.no_grad():
            for cond in conditions:
                for i in range(50):
                    enc = tokenizer(texts_fmt[cond][i], return_tensors="pt", truncation=True, max_length=512)
                    enc = {k: v.to(first_device) for k, v in enc.items()}
                    _ = model(**enc); _nt += enc['input_ids'].shape[1]; del enc, _
    finally:
        for h in _mh: h.remove()
    neuron_mean_acts = (_sum / _nt).to(model.dtype)

from contextlib import contextmanager
@contextmanager
def ablate(nd):
    H = []
    def mk(l, js):
        jt = torch.tensor(js, device=first_device)
        def hook(m, args):
            x = args[0].clone(); x[..., jt] = neuron_mean_acts[l, jt]; return (x,)+args[1:]
        return hook
    try:
        for l, js in nd.items():
            H.append(model.model.layers[l].mlp.down_proj.register_forward_pre_hook(mk(l, list(js))))
        yield
    finally:
        for h in H: h.remove()

def pR(lps):
    R = np.where(sign > 0, lps[:,0], lps[:,1]); U = np.where(sign > 0, lps[:,1], lps[:,0])
    return 1.0/(1.0+np.exp(-(R-U)))
im = lambda v: np.array([v[items_arr==it].mean() for it in np.unique(items_arr)])

# ── baselines ──
print("\n  baselines...")
lm0 = compute_lps_unbatched(texts_fmt['B_cult'], 400, "base/match")
lu0 = compute_lps_unbatched(texts_fmt['B_unrel'], 400, "base/mism")
lnB0 = compute_lps_unbatched(neutralB, 400, "base/neutralB")
pRm0, pRu0 = pR(lm0), pR(lu0)
SnB0 = S_from_lps(lnB0)
base_contrast = (pRm0 - pRu0).mean()
print(f"  baseline: P(R) match {pRm0.mean():.4f} mism {pRu0.mean():.4f} contrast {base_contrast:+.4f}")

def one(nd):
    with ablate(nd):
        lm = compute_lps_unbatched(texts_fmt['B_cult'], None)
        lu = compute_lps_unbatched(texts_fmt['B_unrel'], None)
        ln = compute_lps_unbatched(neutralB, None)
    pRm, pRu = pR(lm), pR(lu)
    dP = im(pRm) - im(pRm0); t, p = ttest_1samp(dP, 0)
    pct_c = ((pRm-pRu).mean() - base_contrast)/abs(base_contrast)*100
    SnB = S_from_lps(ln)
    pct_n = (SnB.mean() - SnB0.mean())/abs(SnB0.mean())*100
    return dict(dP=dP.mean(), t=t, p=p, pct_c=pct_c, pct_n=pct_n,
                pRm=pRm.mean(), pRu=pRu.mean())

sweep = {}
for K in K_SWEEP:
    POS = to_dict(pos_rank[:K]); NEG = to_dict(neg_rank[:K])
    print(f"\n  K={K}  (POS {sum(len(v) for v in POS.values())} / NEG {sum(len(v) for v in NEG.values())})")
    sweep[K] = {'POS': one(POS), 'NEG': one(NEG),
                'POS_set': POS, 'NEG_set': NEG}

print("\n" + "=" * 80)
print(f"  K-SWEEP — {CFG['label']}   (cultural P(R) 847 ; neutral control = split B)")
print("=" * 80)
print(f"  {'K':>3} | {'POS ΔP(R)':>10} {'contr%':>7} {'neutB%':>7} | {'NEG ΔP(R)':>10} {'contr%':>7} {'neutB%':>7}")
for K in K_SWEEP:
    P, N = sweep[K]['POS'], sweep[K]['NEG']
    pp = '*' if P['p'] < .001 else ' '; np_ = '*' if N['p'] < .001 else ' '
    print(f"  {K:3d} | {P['dP']:+10.4f}{pp}{P['pct_c']:7.1f}{P['pct_n']:7.1f} | "
          f"{N['dP']:+10.4f}{np_}{N['pct_c']:7.1f}{N['pct_n']:7.1f}")
print("\n  want: POS ΔP(R)<0, NEG ΔP(R)>0 (symmetry); neutB% small (specific).")
print("  pick the K where directional effect is large while neutB% stays low (the knee).")

neuron_store['sweep'] = {K: {'POS': sweep[K]['POS'], 'NEG': sweep[K]['NEG']} for K in K_SWEEP}
neuron_store['baseline'] = {'pRm': float(pRm0.mean()), 'pRu': float(pRu0.mean()),
                            'SnB': float(SnB0.mean())}
_npkl = OUTPUT_DIR / f"results_{ACTIVE_MODEL}_instruct_mlp_neuron_filtered.pkl"
with open(_npkl, "wb") as f: pickle.dump(neuron_store, f)
print(f"\n  saved sweep to {_npkl}")

## Reading

Per model, the sweep table shows, for each K: POS/NEG ΔP(R) (want POS<0, NEG>0),
contrast %, and **neutB%** (neutral shift on the independent split B). Pick the K
at the *knee*: directional effect large while neutB% stays low. Compare the knee
across the 4 models — expect Mistral cleanest, Nemo worst (diffuse, sign-mixed
late MLPs). The filter should pull Nemo's neutB% down vs the unfiltered run; if it
doesn't, that's the honest finding that even Nemo's cleanest neurons aren't specific.